# GIS-Based School Site Selection in Lahore, Pakistan
### A Business-Oriented Spatial Feasibility Framework

**Author:** Muhammad Abdul Aleem | **Roll No:** MSDS25022  
**Course:** Spatial Data Science — Spring 2026  
**Institution:** Information Technology University (ITU), Lahore

---

## Project Overview
This project applies **GIS**, **Kernel Density Estimation (KDE)**, **Spatial Weight Matrices**,
and **Multi-Criteria Decision Analysis (MCDA)** to identify the most profitable locations
in Lahore for opening a new private school, and estimate the required investment.

| Research Question | Answer |
|---|---|
| **Where** to open? | Spatially optimal zones via MCDA |
| **How much** to invest? | Break-even analysis at PKR 3,000/student fee |

## Data Sources
| Dataset | Source | Records |
|---|---|---|
| Lahore boundary | GADM Level 3 | 1 polygon |
| Government schools | Punjab PITB | 1,465 schools |
| Private/mixed schools | OpenStreetMap | 472 schools |
| Population grid | WorldPop 2024 (100m) | Raster |
| Road network | OSMnx | 144,676 nodes |
| Property rates | Zameen.com / Graana (May 2026) | 12 areas |
---

## Section 0 — Imports & Global Configuration

In [ ]:
# ─────────────────────────────────────────────────────────────────
# IMPORTS
# Standard geospatial and scientific computing libraries
# ─────────────────────────────────────────────────────────────────
import os
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import geopandas as gpd

import rasterio
from rasterio.mask  import mask        # clip raster to polygon
from rasterio.features import geometry_mask  # burn polygon onto array
from rasterio.transform import from_bounds   # create affine transform

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.cm import ScalarMappable
from matplotlib.patches import FancyBboxPatch

from scipy.stats    import gaussian_kde         # KDE on point data
from scipy.ndimage  import gaussian_filter      # smoothing on rasters
from scipy.spatial  import cKDTree             # nearest-neighbour search
from scipy.interpolate import griddata          # spatial interpolation

from shapely.geometry import mapping, Point
import osmnx as ox

print("✅ All libraries imported successfully")
print(f"  NumPy   : {np.__version__}")
print(f"  GeoPandas: {gpd.__version__}")

# ─────────────────────────────────────────────────────────────────
# GLOBAL GRID SETTINGS — all layers use the same 500×500 grid
# Higher = crisper maps but slower compute
# ─────────────────────────────────────────────────────────────────
GRID_SIZE = 500   # 500 × 500 pixel grid over Lahore

## Section 1 — File Paths (Relative — No Modification Needed)
All paths use `os.getcwd()`. Place data files in the `Data/` folder next to this notebook.

In [ ]:
# ─────────────────────────────────────────────────────────────────
# PATHS — fully relative, works on any machine without modification
# ─────────────────────────────────────────────────────────────────
BASE       = os.getcwd()
DATA_DIR   = os.path.join(BASE, 'Data')
OUTPUT_DIR = os.path.join(BASE, 'Outputs')
os.makedirs(OUTPUT_DIR, exist_ok=True)

BOUNDARY_PATH = os.path.join(DATA_DIR, 'gadm41_PAK_3.json')
OSM_SCHOOLS   = os.path.join(DATA_DIR, 'lahore_schools.geojson')
PITB_SCHOOLS  = os.path.join(DATA_DIR, 'Schools.csv')
WORLDPOP_PATH = os.path.join(DATA_DIR, 'pak_pop_2024_CN_100m_R2025A_v1.tif')

print("File availability check:")
for label, path in [
    ("Boundary (GADM)",  BOUNDARY_PATH),
    ("Schools (OSM)",    OSM_SCHOOLS),
    ("Schools (PITB)",   PITB_SCHOOLS),
    ("WorldPop 2024",    WORLDPOP_PATH),
]:
    status = "✅ Found" if os.path.exists(path) else "❌ NOT FOUND — place file in Data/"
    print(f"  {status} — {label}")

## Section 2 — Lahore Boundary (GADM Level 3)

In [ ]:
# ─────────────────────────────────────────────────────────────────
# STEP 1: Load the GADM JSON which contains all Pakistan districts
# STEP 2: Filter to Lahore only using the NAME_3 column
# STEP 3: Reproject to UTM Zone 42N (EPSG:32642)
#         → measures in metres, required for distance-based analysis
# ─────────────────────────────────────────────────────────────────
pakistan   = gpd.read_file(BOUNDARY_PATH)
lahore     = pakistan[pakistan["NAME_3"] == "Lahore"].copy()
lahore_utm = lahore.to_crs(epsg=32642)

print(f"Lahore boundary loaded")
print(f"  Original CRS : {lahore.crs}  (degrees)")
print(f"  Projected CRS: {lahore_utm.crs}  (metres)")

# Extract bounding box in UTM — used by every subsequent layer
MINX, MINY, MAXX, MAXY = lahore_utm.total_bounds
print(f"  UTM bounds   : X [{MINX:.0f}, {MAXX:.0f}]  Y [{MINY:.0f}, {MAXY:.0f}]")

# ─────────────────────────────────────────────────────────────────
# BUILD THE SHARED 500×500 SPATIAL GRID (UTM coordinates)
# Every analysis layer will be projected onto this same grid.
# Row 0 = southernmost, Row 499 = northernmost (origin="lower")
# ─────────────────────────────────────────────────────────────────
x_vals = np.linspace(MINX, MAXX, GRID_SIZE)
y_vals = np.linspace(MINY, MAXY, GRID_SIZE)    # south → north
XX, YY = np.meshgrid(x_vals, y_vals)           # shape (500, 500)
FLAT_GRID = np.c_[XX.ravel(), YY.ravel()]       # shape (250000, 2)

# ─────────────────────────────────────────────────────────────────
# BOUNDARY MASK — True where pixels are OUTSIDE Lahore
# Used to clip every raster layer cleanly to the boundary
# Key trick: set masked values to NaN, then use cmap.set_bad(alpha=0)
# → transparent outside boundary, no yellow/colour bleed
# ─────────────────────────────────────────────────────────────────
GRID_TRANSFORM = from_bounds(MINX, MINY, MAXX, MAXY, GRID_SIZE, GRID_SIZE)
LAHORE_GEOMS   = [mapping(g) for g in lahore_utm.geometry]
OUTSIDE_MASK   = geometry_mask(
    LAHORE_GEOMS,
    transform  = GRID_TRANSFORM,
    invert     = False,       # True = inside, False = outside
    out_shape  = (GRID_SIZE, GRID_SIZE)
)
# Because our grid has origin="lower", we must flip the mask vertically
OUTSIDE_MASK = np.flipud(OUTSIDE_MASK)

print(f"\nShared {GRID_SIZE}×{GRID_SIZE} grid built over Lahore (UTM 42N)")
print(f"  Pixels inside  Lahore: {(~OUTSIDE_MASK).sum():,}")
print(f"  Pixels outside Lahore: {OUTSIDE_MASK.sum():,}")

# ─────────────────────────────────────────────────────────────────
# HELPER FUNCTION: apply boundary mask + make clean colormap
# ─────────────────────────────────────────────────────────────────
def apply_mask(arr):
    """Return masked array; pixels outside Lahore → NaN (transparent in imshow)."""
    out = arr.copy().astype(float)
    out[OUTSIDE_MASK] = np.nan
    return out

def get_cmap(name):
    """Return a copy of a colormap with bad/NaN values set to transparent."""
    cmap = plt.get_cmap(name).copy()
    cmap.set_bad(alpha=0)   # ← this is the fix for boundary bleeding
    return cmap

# Common imshow kwargs so every map uses the same settings
IMSHOW_KW = dict(
    extent = [MINX, MAXX, MINY, MAXY],
    origin = "lower",     # row 0 = south (consistent with our grid)
    alpha  = 0.90,
    zorder = 2
)

# ─────────────────────────────────────────────────────────────────
# PLOT — Lahore boundary
# ─────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(7, 7))
lahore_utm.plot(ax=ax, color="lightblue", edgecolor="black", linewidth=2)
ax.set_title("Lahore District Boundary — GADM Level 3", fontsize=13, fontweight='bold')
ax.axis("off")
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "01_lahore_boundary.png"), dpi=150, bbox_inches='tight')
plt.show()
print("Saved → 01_lahore_boundary.png")

## Section 3 — School Data: PITB Government + OSM Private (Dual Source)

In [ ]:
# ─────────────────────────────────────────────────────────────────
# OSM PRIVATE/MIXED SCHOOLS
# Downloaded via Overpass Turbo (amenity=school) for Lahore
# ─────────────────────────────────────────────────────────────────
osm_raw    = gpd.read_file(OSM_SCHOOLS)
osm_utm    = osm_raw.to_crs(epsg=32642)
osm_lahore = gpd.clip(osm_utm, lahore_utm).copy()
osm_lahore["source"] = "OSM (Private/Mixed)"
print(f"OSM schools loaded and clipped to Lahore: {len(osm_lahore)} records")

# ─────────────────────────────────────────────────────────────────
# PITB GOVERNMENT SCHOOLS (Punjab Information Technology Board)
# 48,195 schools across Punjab — filter to Lahore bounding box
# ─────────────────────────────────────────────────────────────────
def classify_school(name):
    """Classify government schools by level based on naming convention."""
    n = str(name).upper()
    if any(x in n for x in ['GHSS','GGHSS','GHS','GGHS']): return 'Secondary'
    if any(x in n for x in ['GPS', 'GGPS']):               return 'Primary'
    if any(x in n for x in ['GMS', 'GGMS']):               return 'Middle'
    if any(x in n for x in ['GGES','GES']):                return 'Elementary'
    return 'Other Govt'

pitb_raw = pd.read_csv(PITB_SCHOOLS, low_memory=False)

# Geographic filter: Lahore approximate bounding box (WGS84 degrees)
pitb_lhr_df = pitb_raw[
    (pitb_raw['latitude']  >= 31.20) & (pitb_raw['latitude']  <= 31.70) &
    (pitb_raw['longitude'] >= 74.00) & (pitb_raw['longitude'] <= 74.60)
].copy()
pitb_lhr_df['school_type'] = pitb_lhr_df['name'].apply(classify_school)

# Convert to GeoDataFrame, reproject, clip to exact boundary
pitb_lahore = gpd.GeoDataFrame(
    pitb_lhr_df,
    geometry = gpd.points_from_xy(pitb_lhr_df['longitude'], pitb_lhr_df['latitude']),
    crs = "EPSG:4326"
).to_crs(epsg=32642)
pitb_lahore = gpd.clip(pitb_lahore, lahore_utm).copy()
pitb_lahore["source"] = "PITB (Government)"

print(f"PITB schools loaded and clipped to Lahore: {len(pitb_lahore)} records")
print(f"  School type breakdown:")
for stype, cnt in pitb_lahore['school_type'].value_counts().items():
    print(f"    {stype:<15}: {cnt}")

# ─────────────────────────────────────────────────────────────────
# COMBINED DATASET — both sources merged into one GeoDataFrame
# ─────────────────────────────────────────────────────────────────
all_schools = gpd.GeoDataFrame(
    pd.concat([osm_lahore[['geometry','source']], pitb_lahore[['geometry','source']]],
              ignore_index=True),
    geometry = 'geometry', crs = "EPSG:32642"
)
print(f"\nCombined school dataset: {len(all_schools):,} schools")
print(f"  OSM private/mixed: {len(osm_lahore):,}")
print(f"  PITB government  : {len(pitb_lahore):,}")

# Save processed PITB data for submission
pitb_lahore[['name','school_type','source','geometry']].to_file(
    os.path.join(OUTPUT_DIR, 'pitb_schools_lahore.geojson'), driver='GeoJSON')

# ─────────────────────────────────────────────────────────────────
# PLOT — Dual panel: source comparison + type breakdown
# ─────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# Panel A: Both sources overlaid
lahore_utm.plot(ax=axes[0], color="whitesmoke", edgecolor="black", linewidth=1.5)
pitb_lahore.plot(ax=axes[0], color="royalblue", markersize=2.5, alpha=0.6,
                 label=f"PITB Govt (n={len(pitb_lahore):,})")
osm_lahore.plot(ax=axes[0],  color="crimson",    markersize=3.5, alpha=0.75,
                label=f"OSM Private (n={len(osm_lahore):,})")
axes[0].set_title("School Coverage: OSM + PITB Combined\n"
                  "Blue = Government | Red = Private/Mixed", fontsize=11, fontweight='bold')
axes[0].legend(fontsize=9, loc='lower right')
axes[0].axis("off")

# Panel B: PITB school types
type_colors = {'Primary':'#2ecc71','Secondary':'#e74c3c',
               'Elementary':'#f39c12','Middle':'#9b59b6','Other Govt':'#95a5a6'}
lahore_utm.plot(ax=axes[1], color="whitesmoke", edgecolor="black", linewidth=1.5)
for stype, color in type_colors.items():
    sub = pitb_lahore[pitb_lahore['school_type'] == stype]
    if len(sub) > 0:
        sub.plot(ax=axes[1], color=color, markersize=2.5, alpha=0.7,
                 label=f"{stype} ({len(sub):,})")
axes[1].set_title("PITB Schools by Type\n(1,465 Official Punjab Education Dept Records)",
                  fontsize=11, fontweight='bold')
axes[1].legend(fontsize=9, loc='lower right')
axes[1].axis("off")

plt.suptitle("Dual-Source School Dataset — OSM + PITB", fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "02_dual_school_dataset.png"), dpi=150, bbox_inches='tight')
plt.show()
print("Saved → 02_dual_school_dataset.png")

## Section 4 — Road Network (OSMnx)

In [ ]:
# ─────────────────────────────────────────────────────────────────
# OSMnx downloads the driveable road network for any polygon.
# We convert Lahore boundary back to WGS84 (lat/lon) because
# OSMnx requires geographic coordinates, not projected UTM.
# ─────────────────────────────────────────────────────────────────
lahore_wgs84   = lahore_utm.to_crs(epsg=4326)
lahore_polygon = lahore_wgs84.geometry.iloc[0]   # shapely Polygon

print("Downloading Lahore road network from OpenStreetMap...")
G = ox.graph_from_polygon(lahore_polygon, network_type="drive")

print(f"✅ Road network downloaded successfully")
print(f"  Nodes (intersections): {len(G.nodes):,}")
print(f"  Edges (road segments) : {len(G.edges):,}")

# Save to GraphML for reuse in accessibility analysis
ox.save_graphml(G, os.path.join(OUTPUT_DIR, "lahore_road_network.graphml"))

# ─────────────────────────────────────────────────────────────────
# PLOT — Road network
# ─────────────────────────────────────────────────────────────────
fig, ax = ox.plot_graph(G, figsize=(8, 8), node_size=0,
                        edge_color="#333333", edge_linewidth=0.25,
                        bgcolor="lightyellow", show=False, close=False)
ax.set_title(f"Lahore Road Network — OSMnx\n"
             f"{len(G.nodes):,} intersections | {len(G.edges):,} road segments",
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "03_road_network.png"), dpi=150, bbox_inches='tight')
plt.show()
print("Saved → 03_road_network.png")

## Section 5 — WorldPop 2024 Population Raster

In [ ]:
# ─────────────────────────────────────────────────────────────────
# WorldPop provides a gridded population surface for Pakistan.
# Each pixel = 100m × 100m area with estimated population count.
# Steps:
#   1. Open the Pakistan-wide raster
#   2. Match CRS to raster before clipping
#   3. Clip (mask) to Lahore boundary → out_image, out_transform
#   4. Remove nodata values (stored as large negatives)
# ─────────────────────────────────────────────────────────────────
with rasterio.open(WORLDPOP_PATH) as src:
    print(f"Raster CRS        : {src.crs}")
    print(f"Pixel resolution  : {src.res}° ≈ 100m per pixel")
    print(f"Full raster size  : {src.shape[0]:,} rows × {src.shape[1]:,} cols (all Pakistan)")

    # Reproject Lahore boundary to match raster CRS before clipping
    lahore_raster_crs = lahore.to_crs(src.crs)

    # crop=True shrinks the output to the smallest enclosing box
    out_image, out_transform = mask(src, lahore_raster_crs.geometry, crop=True)

out_image = out_image.astype("float32")
out_image[out_image < 0] = np.nan    # replace nodata flag values with NaN

print(f"\nClipped to Lahore:")
print(f"  Raster shape : {out_image.shape[1]:,} rows × {out_image.shape[2]:,} cols")
print(f"  Min population per cell : {np.nanmin(out_image):.1f}")
print(f"  Max population per cell : {np.nanmax(out_image):.1f}")
print(f"  Mean population per cell: {np.nanmean(out_image):.1f}")

# ─────────────────────────────────────────────────────────────────
# REPROJECT POPULATION RASTER → shared 500×500 grid
# The raster has its own resolution; we resize it to our grid.
# IMPORTANT: rasterio rasters are north-up (row 0 = northernmost).
# Our grid is south-up (row 0 = southernmost, origin="lower").
# Fix: flip vertically with np.flipud() after resizing.
# ─────────────────────────────────────────────────────────────────
from scipy.ndimage import zoom as nd_zoom

pop_raw = out_image[0].copy()                     # extract band 1
pop_raw = np.nan_to_num(pop_raw, nan=0.0)         # replace NaN with 0

# Compute zoom factors to resize from raster shape → (GRID_SIZE, GRID_SIZE)
zr = GRID_SIZE / pop_raw.shape[0]
zc = GRID_SIZE / pop_raw.shape[1]
pop_grid = nd_zoom(pop_raw, (zr, zc), order=1)    # bilinear resize

# Flip vertically: raster is north-up, our grid is south-up
pop_grid = np.flipud(pop_grid)

# Normalize to 0–1 scale for MCDA
pop_norm = pop_grid / pop_grid.max()

print(f"\nPopulation grid reprojected to {GRID_SIZE}×{GRID_SIZE}")
print(f"  After flip & normalize: min={pop_norm.min():.3f}, max={pop_norm.max():.3f}")

# ─────────────────────────────────────────────────────────────────
# PLOT — Population density heatmap
# ─────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 8))
lahore_utm.plot(ax=ax, color="whitesmoke", edgecolor="black", linewidth=2, zorder=1)
img = ax.imshow(apply_mask(pop_grid), cmap=get_cmap("YlOrRd"), **IMSHOW_KW)
lahore_utm.plot(ax=ax, color="none", edgecolor="black", linewidth=2, zorder=3)
plt.colorbar(img, ax=ax, label="Population per 100m cell", shrink=0.7)
ax.set_title("WorldPop 2024 — Lahore Population Density\n(100m × 100m resolution)",
             fontsize=12, fontweight='bold')
ax.axis("off")
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "04_worldpop_density.png"), dpi=150, bbox_inches='tight')
plt.show()
print("Saved → 04_worldpop_density.png")

## Section 6 — Data Limitations & Quality Notes

**OSM Schools (472):** Private and mixed schools from OpenStreetMap. Coverage is denser in the urban core; incomplete in southern rural tehsils (Raiwind, Manga Mandi). Used as proxy for private sector supply.

**PITB Schools (1,465):** Official Punjab government school registry from PITB. Covers all tehsils including rural south Lahore. Together with OSM, the combined dataset of **1,937 schools** is far more comprehensive than either alone.

**WorldPop 2024:** 100m resolution gridded population. Max density 345.7 persons/cell. All negative nodata values removed.

**Road Network (OSMnx):** 144,676 nodes, 376,672 edges. Comprehensive for major and secondary roads. Internal colony lanes may be underrepresented.

**Property Data:** Commercial building rents collected from Zameen.com and Graana.com market reports, May 2026. Standard reference size: 10-Marla commercial building suitable for a school.

## Section 7 — Real Property Cost Data (Zameen.com + Graana.com, May 2026)

In [ ]:
# ─────────────────────────────────────────────────────────────────
# REAL COMMERCIAL RENTAL RATES — verified from published sources
# Reference property: 10-Marla commercial building for school use
# Sources:
#   • Zameen.com building listings (Gulberg, DHA, Cantt, Johar Town)
#   • Graana.com area market reports
#   • Eastern Housing Commercial Market Report 2025
# ─────────────────────────────────────────────────────────────────
zameen_data = {
    "Area": [
        "Gulberg",           # Zameen: PKR 200k–500k (Eastern Housing report)
        "DHA Phase 1-5",     # Zameen/Graana: PKR 118k–200k
        "Johar Town",        # Eastern Housing: PKR 50k–150k
        "Model Town",        # Graana: 1 Kanal ≈ 180k → 10 Marla ≈ 120k
        "Bahria Town",       # Graana: commercial from 100k
        "Township",          # Peri-urban market avg: 40k–80k
        "Faisal Town",       # Market avg: 50k–90k
        "Wapda Town",        # Graana avg: 35k–70k
        "Allama Iqbal Town", # Market avg: 50k–90k
        "Raiwind Road",      # Peri-urban: 30k–60k
        "Cantt",             # Zameen listings: 50k–180k
        "Iqbal Town"         # Market avg: 45k–80k
    ],
    "Min_Rent_PKR": [200000, 118000,  50000, 120000, 100000,
                      40000,  50000,  35000,  50000,  30000,  50000,  45000],
    "Max_Rent_PKR": [500000, 200000, 150000, 180000, 120000,
                      80000,  90000,  70000,  90000,  60000, 180000,  80000],
    # Geographic centroids for spatial interpolation
    "Lat": [31.5204, 31.4697, 31.4697, 31.5149, 31.3635,
            31.4792, 31.4560, 31.4463, 31.5023, 31.3825, 31.5497, 31.5100],
    "Lon": [74.3587, 74.4025, 74.2799, 74.3337, 74.2073,
            74.2637, 74.3016, 74.2776, 74.3429, 74.3750, 74.4085, 74.3250],
    "Zone_Type": [
        "High-Cost Urban",   "High-Cost Urban",  "Mid-Cost Urban",   "Mid-Cost Urban",
        "Low-Cost Suburban", "Low-Cost Suburban","Low-Cost Suburban","Low-Cost Suburban",
        "Low-Cost Suburban", "Peri-Urban",        "High-Cost Urban",  "Low-Cost Suburban"
    ],
}

zameen_df = pd.DataFrame(zameen_data)
zameen_df["Avg_Rent_PKR"]   = ((zameen_df["Min_Rent_PKR"] + zameen_df["Max_Rent_PKR"]) / 2).astype(int)
zameen_df["Annual_Cost_PKR"]= zameen_df["Avg_Rent_PKR"] * 12
# Cost score: 1 = cheapest (best for investor), 0 = most expensive
zameen_df["Cost_Score"]     = 1 - (zameen_df["Avg_Rent_PKR"] / zameen_df["Avg_Rent_PKR"].max())

# Convert to GeoDataFrame (WGS84 → UTM) for spatial operations
zameen_gdf = gpd.GeoDataFrame(
    zameen_df,
    geometry = gpd.points_from_xy(zameen_df["Lon"], zameen_df["Lat"]),
    crs = "EPSG:4326"
).to_crs(epsg=32642)

zameen_df.to_csv(os.path.join(OUTPUT_DIR, "property_costs_zameen.csv"), index=False)

print("Commercial Rental Data — Lahore (Zameen.com + Graana.com, May 2026)")
print("Reference: 10-Marla commercial building")
print("=" * 65)
print(zameen_df[["Area","Min_Rent_PKR","Max_Rent_PKR","Avg_Rent_PKR","Zone_Type"]].to_string(index=False))

# ─────────────────────────────────────────────────────────────────
# PLOT — Rental rates bar chart
# ─────────────────────────────────────────────────────────────────
zone_colors = {"High-Cost Urban":"#e74c3c", "Mid-Cost Urban":"#f39c12",
               "Low-Cost Suburban":"#2ecc71", "Peri-Urban":"#3498db"}
bar_colors  = [zone_colors[z] for z in zameen_df["Zone_Type"]]

fig, ax = plt.subplots(figsize=(12, 6))
bars = ax.barh(zameen_df["Area"], zameen_df["Avg_Rent_PKR"] / 1000,
               color=bar_colors, edgecolor="white", height=0.65)
ax.set_xlabel("Average Monthly Rent — 10 Marla Commercial Building (PKR thousands)", fontsize=11)
ax.set_title("Verified Commercial Rental Rates by Area — Lahore\n"
             "Source: Zameen.com + Graana.com + Eastern Housing Market Report (May 2026)", fontsize=12)
ax.invert_yaxis()
for bar, row in zip(bars, zameen_df.itertuples()):
    ax.text(bar.get_width() + 2, bar.get_y() + bar.get_height() / 2,
            f"PKR {row.Avg_Rent_PKR:,}", va='center', fontsize=8.5)
legend_patches = [mpatches.Patch(color=v, label=k) for k, v in zone_colors.items()]
ax.legend(handles=legend_patches, title="Zone Type", fontsize=9, loc='lower right')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "05_property_costs.png"), dpi=150, bbox_inches='tight')
plt.show()
print("Saved → 05_property_costs.png")

## Section 8 — KDE: School Competition Surface (OSM + PITB Combined)

In [ ]:
# ─────────────────────────────────────────────────────────────────
# KERNEL DENSITY ESTIMATION — School locations
# Purpose: Find WHERE schools are clustered (high competition zones)
#
# Method: scipy gaussian_kde places a smooth "hill" over each school
# location. Hills overlap and sum to produce a continuous surface.
# Bandwidth (bw_method): controls smoothness — 0.10 = moderately smooth
# We evaluate KDE at every point of our 500×500 shared grid.
# ─────────────────────────────────────────────────────────────────

# Extract UTM coordinates from combined school dataset
school_x = all_schools.geometry.x.values
school_y = all_schools.geometry.y.values
school_coords = np.vstack([school_x, school_y])   # shape (2, N)

# Fit KDE to school locations
kde_schools = gaussian_kde(school_coords, bw_method=0.10)

# Evaluate on our 500×500 grid (FLAT_GRID shape: (250000, 2))
print(f"Evaluating KDE on {GRID_SIZE}×{GRID_SIZE} grid... (may take 30–60 seconds)")
school_density = kde_schools(FLAT_GRID.T).reshape(GRID_SIZE, GRID_SIZE)

# Normalize to 0–1 for comparability
school_norm = school_density / school_density.max()

print(f"✅ School competition KDE complete")
print(f"  Schools used : {len(all_schools):,} (OSM {len(osm_lahore)} + PITB {len(pitb_lahore)})")
print(f"  Grid shape   : {school_density.shape}")
print(f"  Value range  : [{school_density.min():.2e}, {school_density.max():.2e}]")

# ─────────────────────────────────────────────────────────────────
# PLOT — School competition KDE map
# Note: apply_mask() + get_cmap() with set_bad(alpha=0)
# ensures NO colour bleeds outside the Lahore boundary
# ─────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 9))
lahore_utm.plot(ax=ax, color="whitesmoke", edgecolor="black", linewidth=2, zorder=1)

img = ax.imshow(apply_mask(school_norm), cmap=get_cmap("YlOrRd"), **IMSHOW_KW)

# Overlay school dots for validation
ax.scatter(school_x, school_y, s=1.0, c="navy", alpha=0.20, zorder=3, label=f"Schools (n={len(all_schools):,})")
lahore_utm.plot(ax=ax, color="none", edgecolor="black", linewidth=2, zorder=4)

plt.colorbar(img, ax=ax, label="School Density — KDE (OSM + PITB combined)", shrink=0.7)
ax.set_title(f"KDE — School Competition Surface\n"
             f"Red = Very High Competition | Yellow = Low Competition (n={len(all_schools):,})",
             fontsize=13, fontweight='bold')
ax.legend(fontsize=9, loc='lower right')
ax.axis("off")
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "06_kde_school_competition.png"), dpi=150, bbox_inches='tight')
plt.show()
print("Saved → 06_kde_school_competition.png")

## Section 9 — KDE: Population Demand Surface (WorldPop 2024)

In [ ]:
# ─────────────────────────────────────────────────────────────────
# POPULATION DEMAND SURFACE
# Purpose: Find WHERE children are concentrated (high demand zones)
#
# Method: WorldPop raster gives population per 100m cell.
# We apply a moderate Gaussian smoothing (sigma=3) to create a
# continuous surface. Sigma=3 means influence radius ≈ 300m.
# Lower sigma = crisper map. Higher sigma = smoother but blurrier.
#
# IMPORTANT coordinate fix:
#   - WorldPop raster: row 0 = NORTH (raster convention)
#   - Our grid: row 0 = SOUTH (Cartesian, origin="lower")
#   - pop_grid was already flipped in Section 5 → consistent now
# ─────────────────────────────────────────────────────────────────

# Apply light smoothing to population grid (already on 500×500 grid)
# sigma=3 → moderate smoothing, preserves spatial detail
pop_smooth = gaussian_filter(pop_grid, sigma=3)

# Normalize to 0–1
pop_smooth_norm = pop_smooth / pop_smooth.max()

print(f"Population demand surface prepared")
print(f"  Smoothing sigma : 3 pixels (≈300m radius)")
print(f"  Value range     : [{pop_smooth_norm.min():.3f}, {pop_smooth_norm.max():.3f}]")

# ─────────────────────────────────────────────────────────────────
# PLOT — Population demand map
# ─────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 9))
lahore_utm.plot(ax=ax, color="whitesmoke", edgecolor="black", linewidth=2, zorder=1)
img = ax.imshow(apply_mask(pop_smooth_norm), cmap=get_cmap("YlOrRd"), **IMSHOW_KW)
lahore_utm.plot(ax=ax, color="none", edgecolor="black", linewidth=2, zorder=3)
plt.colorbar(img, ax=ax, label="Population Density — KDE Smoothed (WorldPop 2024)", shrink=0.7)
ax.set_title("KDE — Child Population Demand Surface\n"
             "Red = High Student Demand | Yellow = Low Demand",
             fontsize=13, fontweight='bold')
ax.axis("off")
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "07_kde_population_demand.png"), dpi=150, bbox_inches='tight')
plt.show()
print("Saved → 07_kde_population_demand.png")

## Section 10 — Underserved Zone Detection (Demand − Competition)

In [ ]:
# ─────────────────────────────────────────────────────────────────
# UNDERSERVED ZONES = Areas with HIGH population but FEW schools
#
# Formula: Underserved Score = Population Demand − School Competition
#
# Both surfaces are on the same 500×500 grid and normalized 0–1.
# A high underserved score means:
#   → Many children live here (high demand)
#   → But few schools exist nearby (low competition)
#   → This is the best business opportunity
#
# np.clip(..., 0, None) removes negatives (already well-served areas).
# ─────────────────────────────────────────────────────────────────

# Compute underserved score
underserved = np.clip(pop_smooth_norm - school_norm, 0, None)
print(f"Underserved score range: [{underserved.min():.3f}, {underserved.max():.3f}]")
print(f"  Pixels with opportunity score > 0.3: {(underserved > 0.3).sum():,}")
print(f"  Pixels with opportunity score > 0.5: {(underserved > 0.5).sum():,}")

# ─────────────────────────────────────────────────────────────────
# PLOT — Three-panel comparison map
# Panel 1: Population demand surface
# Panel 2: School competition surface
# Panel 3: Underserved opportunity zones
# ─────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(21, 7))

panels = [
    (pop_smooth_norm, "YlOrRd",   "Population Demand\n(Where children are — WorldPop 2024)",     "Demand Score (0–1)"),
    (school_norm,     "YlOrRd",   "School Competition\n(OSM + PITB combined — 1,937 schools)",   "Competition Score (0–1)"),
    (underserved,     "RdYlGn_r", "Underserved Zones\n(Demand − Competition = Opportunity)",     "Opportunity Score (0–1)"),
]

for ax, (data, cmap_name, title, cbar_label) in zip(axes, panels):
    lahore_utm.plot(ax=ax, color="whitesmoke", edgecolor="black", linewidth=1.5, zorder=1)
    im = ax.imshow(apply_mask(data), cmap=get_cmap(cmap_name), **IMSHOW_KW)
    lahore_utm.plot(ax=ax, color="none", edgecolor="black", linewidth=1.5, zorder=3)
    plt.colorbar(im, ax=ax, shrink=0.75, label=cbar_label)
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.axis("off")

plt.suptitle("Spatial Analysis — School Site Opportunity Detection\n"
             "Red = HIGH opportunity | Green = well-served (avoid opening here)",
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "08_underserved_zones.png"), dpi=150, bbox_inches='tight')
plt.show()
print("✅ Underserved zone map generated")
print("Saved → 08_underserved_zones.png")

## Section 11 — MCDA Framework

**Multi-Criteria Decision Analysis (MCDA)** combines four independent spatial layers  
into a single composite suitability score for every location in Lahore.

| # | Criterion | Data Source | Weight | Logic |
|---|---|---|---|---|
| 1 | Child Population Density | WorldPop 2024 | **35%** | More children = more potential students |
| 2 | School Gap (Distance-based) | OSM + PITB via KD-Tree | **30%** | Farther from existing schools = less competition |
| 3 | Low Property Cost | Zameen.com / Graana | **20%** | Lower rent = lower setup investment |
| 4 | Road Accessibility | OSMnx edge density | **15%** | Better roads = easier for parents to reach |

**Formula:**
```
Suitability = 0.35 × Population + 0.30 × SchoolGap + 0.20 × LowCost + 0.15 × Accessibility
```

In [ ]:
# ─────────────────────────────────────────────────────────────────
# MCDA LAYER 1: Population demand (already computed in Section 9)
# ─────────────────────────────────────────────────────────────────
layer_pop = pop_smooth_norm.copy()   # 0–1, high = many children

# ─────────────────────────────────────────────────────────────────
# MCDA LAYER 2: School Gap using KD-Tree nearest-neighbour distance
#
# Better than KDE-based gap because:
#   - Works for every single pixel, not just where KDE has signal
#   - Gives intuitive metric: "how far is the nearest school?"
#   - No normalization artifacts from a very peaked KDE surface
#
# Method:
#   1. Build a KD-Tree from all school coordinates (fast O(n log n))
#   2. Query nearest school distance for each of 250,000 grid points
#   3. Normalize 0–1; farther away = higher gap score
# ─────────────────────────────────────────────────────────────────
print("Building KD-Tree for school gap analysis...")
school_tree = cKDTree(school_coords.T)   # school_coords is (2,N) → transpose to (N,2)

# Query: for each grid point, distance to nearest school (k=1)
dist_to_nearest, _ = school_tree.query(FLAT_GRID, k=1)
dist_grid = dist_to_nearest.reshape(GRID_SIZE, GRID_SIZE)

# Normalize: 0 = right next to a school, 1 = farthest from any school
layer_gap = dist_grid / dist_grid.max()

print(f"  Max distance to nearest school: {dist_grid.max():.0f} m")
print(f"  Mean distance                 : {dist_grid.mean():.0f} m")

# ─────────────────────────────────────────────────────────────────
# MCDA LAYER 3: Low Property Cost — Zameen.com spatial interpolation
#
# We have 12 area-level cost scores (0–1, 1=cheapest).
# We interpolate them across all 250,000 grid points using
# 'nearest' method → covers 100% of Lahore (no missing zones).
# ─────────────────────────────────────────────────────────────────
print("Interpolating property cost layer across Lahore...")
zam_xy     = np.array([[g.x, g.y] for g in zameen_gdf.geometry])  # shape (12, 2)
zam_scores = zameen_df["Cost_Score"].values

# 'nearest' ensures every grid point gets the cost of its nearest area centroid
# This covers all of Lahore without holes or convex-hull cutoff issues
cost_grid  = griddata(zam_xy, zam_scores, FLAT_GRID, method='nearest').reshape(GRID_SIZE, GRID_SIZE)
layer_cost = np.clip(cost_grid, 0, 1)

print(f"  Cost layer range: [{layer_cost.min():.3f}, {layer_cost.max():.3f}]")
print(f"  NaN pixels: {np.isnan(layer_cost).sum()} (should be 0 with method='nearest')")

# ─────────────────────────────────────────────────────────────────
# MCDA LAYER 4: Road Accessibility — OSMnx edge density
#
# Method: bin all road edge centroids onto the 500×500 grid,
# then smooth the resulting count surface with gaussian_filter.
# High density = many roads = well-connected = high score.
# ─────────────────────────────────────────────────────────────────
print("Computing road accessibility layer from OSMnx edges...")
edges_gdf = ox.graph_to_gdfs(G, nodes=False, edges=True)
edges_utm = edges_gdf.to_crs(epsg=32642)

# Get centroid of each road edge for binning
edge_centroids = np.array([
    [geom.centroid.x, geom.centroid.y]
    for geom in edges_utm.geometry if geom is not None
])

# Convert centroid coordinates to grid indices (column, row)
col_idx = np.clip(((edge_centroids[:,0] - MINX) / (MAXX - MINX) * (GRID_SIZE-1)).astype(int), 0, GRID_SIZE-1)
row_idx = np.clip(((edge_centroids[:,1] - MINY) / (MAXY - MINY) * (GRID_SIZE-1)).astype(int), 0, GRID_SIZE-1)

# Count edges per cell
road_count = np.zeros((GRID_SIZE, GRID_SIZE))
np.add.at(road_count, (row_idx, col_idx), 1)    # row = y, col = x

# Smooth and normalize
road_smooth = gaussian_filter(road_count.astype(float), sigma=10)
layer_road  = road_smooth / road_smooth.max() if road_smooth.max() > 0 else road_smooth

print(f"  Road edges binned : {len(edge_centroids):,}")
print(f"  Access layer range: [{layer_road.min():.3f}, {layer_road.max():.3f}]")

# ─────────────────────────────────────────────────────────────────
# PLOT — All four MCDA input layers (2×2 grid)
# ─────────────────────────────────────────────────────────────────
layer_specs = [
    (layer_pop,  "YlOrRd", "Layer 1: Child Population Demand (35%)\n(Red = many children)"),
    (layer_gap,  "YlGn",   "Layer 2: School Gap — KD-Tree Distance (30%)\n(Green = far from schools = less competition)"),
    (layer_cost, "YlGn",   "Layer 3: Low Property Cost — Zameen.com (20%)\n(Green = cheaper rent = better for investor)"),
    (layer_road, "Blues",  "Layer 4: Road Accessibility — OSMnx (15%)\n(Blue = dense road network = easier to reach)"),
]

fig, axes = plt.subplots(2, 2, figsize=(16, 14))
for ax, (data, cmap_name, title) in zip(axes.flatten(), layer_specs):
    lahore_utm.plot(ax=ax, color="whitesmoke", edgecolor="black", linewidth=1.5, zorder=1)
    im = ax.imshow(apply_mask(data), cmap=get_cmap(cmap_name), **IMSHOW_KW)
    lahore_utm.plot(ax=ax, color="none", edgecolor="black", linewidth=1.5, zorder=3)
    plt.colorbar(im, ax=ax, shrink=0.75, label="Score (0–1)")
    ax.set_title(title, fontsize=11, fontweight='bold')
    ax.axis("off")

plt.suptitle("MCDA Input Layers — Four Criteria for School Site Selection",
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "09_mcda_four_layers.png"), dpi=150, bbox_inches='tight')
plt.show()
print("✅ All four MCDA layers computed")
print("Saved → 09_mcda_four_layers.png")

## Section 12 — MCDA Composite Suitability Map

In [ ]:
# ─────────────────────────────────────────────────────────────────
# COMPUTE WEIGHTED MCDA SCORE
# All layers are already normalized 0–1 and on the same 500×500 grid.
# Weighted sum → normalize → apply boundary mask.
# ─────────────────────────────────────────────────────────────────
WEIGHTS = {"population": 0.35, "school_gap": 0.30, "cost": 0.20, "access": 0.15}

mcda_raw = (
    WEIGHTS["population"] * layer_pop  +
    WEIGHTS["school_gap"] * layer_gap  +
    WEIGHTS["cost"]       * layer_cost +
    WEIGHTS["access"]     * layer_road
)
mcda = mcda_raw / mcda_raw.max()   # normalize final score to 0–1

print("MCDA composite score computed")
print(f"  Weights applied: {WEIGHTS}")
print(f"  Score range    : [{mcda.min():.3f}, {mcda.max():.3f}]")

# Custom colormap: red (low) → orange → yellow → green (high)
cmap_mcda = LinearSegmentedColormap.from_list(
    "mcda", ["#d73027","#fc8d59","#fee090","#91cf60","#1a9850"], N=256
)

# ─────────────────────────────────────────────────────────────────
# PLOT — MCDA suitability map
# Weight legend placed OUTSIDE the axes (right side) using fig.text
# to avoid any overlap with the map itself.
# ─────────────────────────────────────────────────────────────────
fig = plt.figure(figsize=(13, 10))

# Main map axis takes 80% of width; leave 20% right margin for legend
ax  = fig.add_axes([0.02, 0.02, 0.72, 0.94])     # [left, bottom, width, height]
cax = fig.add_axes([0.76, 0.25, 0.02, 0.50])      # colorbar axis

lahore_utm.plot(ax=ax, color="whitesmoke", edgecolor="black", linewidth=2, zorder=1)
img = ax.imshow(apply_mask(mcda), cmap=cmap_mcda, **IMSHOW_KW)
all_schools.plot(ax=ax, color="black", markersize=1, alpha=0.15, zorder=3)
lahore_utm.plot(ax=ax, color="none", edgecolor="black", linewidth=2, zorder=4)

# Colorbar in dedicated axis
cb = fig.colorbar(img, cax=cax)
cb.set_label("MCDA Suitability Score (0 = Poor → 1 = Best)", fontsize=9)

ax.set_title("MCDA Composite Suitability Map\n"
             "Best Locations in Lahore to Open a Private School",
             fontsize=13, fontweight='bold', pad=10)
ax.axis("off")

# ── Weight legend box placed cleanly in right margin ──
weight_text = (
    "MCDA Weights\n"
    "─────────────────\n"
    f"Population Demand : {int(WEIGHTS['population']*100)}%\n"
    f"School Gap        : {int(WEIGHTS['school_gap']*100)}%\n"
    f"Low Property Cost : {int(WEIGHTS['cost']*100)}%\n"
    f"Road Accessibility: {int(WEIGHTS['access']*100)}%\n"
    "─────────────────\n"
    "● Black dots = existing schools"
)
fig.text(0.78, 0.78, weight_text,
         fontsize=9, verticalalignment='top', fontfamily='monospace',
         bbox=dict(boxstyle='round,pad=0.6', facecolor='white',
                   edgecolor='grey', alpha=0.92))

plt.savefig(os.path.join(OUTPUT_DIR, "10_mcda_suitability_map.png"), dpi=150, bbox_inches='tight')
plt.show()
print("✅ MCDA suitability map saved")
print("Saved → 10_mcda_suitability_map.png")

## Section 13 — Break-Even Financial Analysis (PKR 3,000/student fee)

In [ ]:
# ─────────────────────────────────────────────────────────────────
# BREAK-EVEN ANALYSIS
# Assumption: Average school fee = PKR 3,000 per student per month
# Building size: 10 Marla commercial (our reference property)
#
# Monthly fixed cost breakdown:
#   Rent          → from Zameen.com (area-specific)
#   Staff         → 5 teachers × PKR 50,000 avg
#   Utilities     → electricity, water, gas
#   Admin & misc  → stationery, maintenance, miscellaneous
#   Setup amort.  → furniture + renovation ÷ 36 months
#
# Break-even students = Total Monthly Cost ÷ Fee per student
# ─────────────────────────────────────────────────────────────────
FEE_PER_STUDENT = 3000    # PKR per student per month

# Fixed costs (same across all areas)
STAFF_COST    = 250_000   # 5 teachers × PKR 50,000
UTILITIES     =  35_000   # electricity, water, gas
ADMIN_MISC    =  25_000   # stationery, admin, maintenance
SETUP_AMORT   =  50_000   # renovation + furniture ÷ 36 months (monthly share)
FIXED_NON_RENT = STAFF_COST + UTILITIES + ADMIN_MISC + SETUP_AMORT

print(f"Fixed monthly costs (excluding rent) : PKR {FIXED_NON_RENT:,}")
print(f"Fee per student per month            : PKR {FEE_PER_STUDENT:,}")
print()

rows = []
for _, area in zameen_df.iterrows():
    total_monthly  = area["Avg_Rent_PKR"] + FIXED_NON_RENT
    breakeven      = int(np.ceil(total_monthly / FEE_PER_STUDENT))
    annual_cost    = total_monthly * 12
    # Profit = Revenue - Cost at different enrolment levels
    for students in [100, 150, 200]:
        revenue = students * FEE_PER_STUDENT * 12
    rows.append({
        "Area"             : area["Area"],
        "Avg_Rent_PKR"     : area["Avg_Rent_PKR"],
        "Total_Monthly_PKR": total_monthly,
        "Breakeven_Students": breakeven,
        "Annual_Cost_PKR"  : annual_cost,
        "Profit_100st_PKR" : 100 * FEE_PER_STUDENT * 12 - annual_cost,
        "Profit_150st_PKR" : 150 * FEE_PER_STUDENT * 12 - annual_cost,
        "Profit_200st_PKR" : 200 * FEE_PER_STUDENT * 12 - annual_cost,
        "Zone_Type"        : area["Zone_Type"],
        "Lat"              : area["Lat"],
        "Lon"              : area["Lon"],
    })

fin_df = pd.DataFrame(rows).sort_values("Breakeven_Students")
fin_df.to_csv(os.path.join(OUTPUT_DIR, "breakeven_analysis_3000pkr.csv"), index=False)

# ── Print summary table ──
print(f"Break-Even Analysis — Fee: PKR {FEE_PER_STUDENT:,}/student/month")
print("=" * 72)
print(f"{'Area':<22} {'Avg Rent':>12} {'Total/Month':>12} {'Break-Even':>12}")
print("-" * 72)
for _, r in fin_df.iterrows():
    print(f"{r.Area:<22} {r.Avg_Rent_PKR:>12,} {r.Total_Monthly_PKR:>12,} {r.Breakeven_Students:>10} stu")

# ─────────────────────────────────────────────────────────────────
# PLOT 1 — Break-even bar chart
# ─────────────────────────────────────────────────────────────────
be_colors = ["#27ae60" if b <= 120 else "#f39c12" if b <= 180 else "#e74c3c"
             for b in fin_df["Breakeven_Students"]]

fig, axes = plt.subplots(1, 2, figsize=(18, 6))

# Left: break-even students
bars = axes[0].barh(fin_df["Area"], fin_df["Breakeven_Students"],
                    color=be_colors, edgecolor="white", height=0.65)
axes[0].axvline(x=120, color='black', linewidth=2, linestyle='--',
                alpha=0.7, label='Target: 120 students')
axes[0].set_xlabel(f"Students Needed to Break Even/Month (Fee = PKR {FEE_PER_STUDENT:,})", fontsize=10)
axes[0].set_title("Break-Even Enrollment by Area\n"
                  "Green ≤120 | Orange 121–180 | Red >180", fontsize=11)
axes[0].invert_yaxis()
for bar, val in zip(bars, fin_df["Breakeven_Students"]):
    axes[0].text(bar.get_width() + 1, bar.get_y() + bar.get_height()/2,
                 f"{val}", va='center', fontsize=9, fontweight='bold')
axes[0].legend(fontsize=9)

# Right: profit scenario comparison
x_pos = np.arange(len(fin_df))
w     = 0.25
axes[1].bar(x_pos - w, fin_df["Profit_100st_PKR"] / 1e6, w,
            label="100 students", color="#3498db", alpha=0.85)
axes[1].bar(x_pos,     fin_df["Profit_150st_PKR"] / 1e6, w,
            label="150 students", color="#2ecc71", alpha=0.85)
axes[1].bar(x_pos + w, fin_df["Profit_200st_PKR"] / 1e6, w,
            label="200 students", color="#27ae60", alpha=0.85)
axes[1].axhline(0, color='black', linewidth=1)
axes[1].set_xticks(x_pos)
axes[1].set_xticklabels(fin_df["Area"], rotation=35, ha='right', fontsize=8.5)
axes[1].set_ylabel("Annual Profit / Loss (PKR millions)")
axes[1].set_title(f"Profit Scenarios by Area\nFee: PKR {FEE_PER_STUDENT:,}/month | At 100, 150, 200 students",
                  fontsize=11)
axes[1].legend(fontsize=9)

plt.suptitle(f"Financial Feasibility Analysis — Private School in Lahore\n"
             f"Fee: PKR {FEE_PER_STUDENT:,}/student/month | Source: Zameen.com (May 2026)",
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "11_breakeven_analysis.png"), dpi=150, bbox_inches='tight')
plt.show()
print("Saved → 11_breakeven_analysis.png + breakeven_analysis_3000pkr.csv")

## Section 14 — Final Investment Ranking (MCDA + Financial Combined)

In [ ]:
# ─────────────────────────────────────────────────────────────────
# COMBINED INVESTMENT RANKING
# Merges spatial suitability (MCDA) with financial viability.
#
# Approach:
#   1. Extract MCDA score at each area's centroid location
#   2. Compute financial score = 1 − (breakeven / max_breakeven)
#      → lower break-even requirement = better financial score
#   3. Combined Score = 0.60 × MCDA_Spatial + 0.40 × Financial
# ─────────────────────────────────────────────────────────────────

# Extract MCDA score at each Zameen.com area location
mcda_at_areas = []
for _, row in zameen_gdf.iterrows():
    # Convert UTM coordinates → grid indices
    col = int(np.clip((row.geometry.x - MINX) / (MAXX - MINX) * (GRID_SIZE-1), 0, GRID_SIZE-1))
    row_i = int(np.clip((row.geometry.y - MINY) / (MAXY - MINY) * (GRID_SIZE-1), 0, GRID_SIZE-1))
    score = mcda[row_i, col]
    mcda_at_areas.append(float(score) if not np.isnan(score) else 0.0)

zameen_df["MCDA_Score"] = mcda_at_areas

# Merge with financial data
ranked = fin_df.merge(zameen_df[["Area","MCDA_Score"]], on="Area")

# Financial score: lower break-even = better
max_be = ranked["Breakeven_Students"].max()
ranked["Financial_Score"] = 1 - (ranked["Breakeven_Students"] / max_be)

# Final combined score
ranked["Combined_Score"] = 0.60 * ranked["MCDA_Score"] + 0.40 * ranked["Financial_Score"]
ranked = ranked.sort_values("Combined_Score", ascending=False).reset_index(drop=True)
ranked["Rank"] = ranked.index + 1

ranked.to_csv(os.path.join(OUTPUT_DIR, "top_locations_ranked.csv"), index=False)

# ── Print ranking ──
print("FINAL INVESTMENT RANKING — Top School Locations in Lahore")
print("Score = 60% Spatial MCDA + 40% Financial Viability")
print("=" * 78)
print(f"{'Rank':<5} {'Area':<22} {'MCDA':>6} {'Fin':>6} {'Combined':>9} {'Break-even':>12}")
print("-" * 78)
medals = {1:"🥇", 2:"🥈", 3:"🥉"}
for _, r in ranked.iterrows():
    medal = medals.get(int(r.Rank), "  ")
    print(f"{medal} {int(r.Rank):<4} {r.Area:<22} {r.MCDA_Score:>6.3f} "
          f"{r.Financial_Score:>6.3f} {r.Combined_Score:>9.3f} "
          f"{int(r.Breakeven_Students):>10} students")

# ─────────────────────────────────────────────────────────────────
# PLOT — Final investment recommendation map
# Numbered markers + external legend table (no overlapping labels)
# ─────────────────────────────────────────────────────────────────
fig = plt.figure(figsize=(18, 10))

# Layout: map on left (65%), legend table on right (35%)
ax_map = fig.add_axes([0.01, 0.02, 0.60, 0.94])
ax_leg = fig.add_axes([0.63, 0.02, 0.36, 0.94])
ax_leg.axis("off")

# Draw MCDA background on map
lahore_utm.plot(ax=ax_map, color="whitesmoke", edgecolor="black", linewidth=2, zorder=1)
img = ax_map.imshow(apply_mask(mcda), cmap=cmap_mcda, **IMSHOW_KW)
lahore_utm.plot(ax=ax_map, color="none", edgecolor="black", linewidth=2, zorder=3)

# Marker colors by rank tier
def rank_color(rank):
    if rank == 1:   return "gold"
    if rank == 2:   return "silver"
    if rank == 3:   return "#cd7f32"   # bronze
    if rank <= 6:   return "#2ecc71"   # green
    if rank <= 9:   return "#f39c12"   # orange
    return "#e74c3c"                   # red

# Plot numbered circles only (no inline text — table handles labels)
for _, r in ranked.iterrows():
    pt = gpd.GeoDataFrame(
        geometry=[Point(r.Lon, r.Lat)], crs="EPSG:4326"
    ).to_crs(epsg=32642)
    cx, cy = pt.geometry.x.iloc[0], pt.geometry.y.iloc[0]
    color = rank_color(int(r.Rank))

    # Outer white ring for contrast
    ax_map.scatter(cx, cy, s=280, c="white",   zorder=5, edgecolor='black', linewidth=1.5)
    ax_map.scatter(cx, cy, s=200, c=color,     zorder=6, edgecolor='black', linewidth=1.0)
    ax_map.text(cx, cy, str(int(r.Rank)),
                ha='center', va='center', fontsize=7.5, fontweight='bold', zorder=7)

ax_map.set_title("Final Investment Recommendation — Top School Locations in Lahore\n"
                 "Numbered markers = ranked areas | Background = MCDA suitability",
                 fontsize=12, fontweight='bold')
ax_map.axis("off")

# Colorbar
cbar_ax = fig.add_axes([0.605, 0.20, 0.012, 0.55])
cb = fig.colorbar(img, cax=cbar_ax)
cb.set_label("MCDA Score", fontsize=8)

# ── Legend table (right panel) ──
ax_leg.text(0.05, 0.98, "Investment Ranking Summary",
            fontsize=13, fontweight='bold', va='top', transform=ax_leg.transAxes)
ax_leg.text(0.05, 0.93,
            f"Combined Score = 60% Spatial + 40% Financial\nFee: PKR {FEE_PER_STUDENT:,}/student/month",
            fontsize=9, va='top', transform=ax_leg.transAxes, color='grey')

headers = ["Rank", "Area", "Break-Even", "Monthly Cost", "Score"]
col_x   = [0.02,   0.16,   0.52,         0.70,           0.92]
y_start = 0.86
row_h   = 0.063

# Header row
for hdr, cx in zip(headers, col_x):
    ax_leg.text(cx, y_start, hdr, fontsize=9, fontweight='bold',
                va='top', transform=ax_leg.transAxes)
ax_leg.axhline(y=y_start - 0.015, xmin=0.02, xmax=0.98,
               color='black', linewidth=1.2, transform=ax_leg.transAxes)

# Data rows
for i, (_, r) in enumerate(ranked.iterrows()):
    y = y_start - row_h * (i + 1)
    # Alternating row background
    bg_color = "#f8f8f8" if i % 2 == 0 else "white"
    rect = FancyBboxPatch((0.01, y - 0.005), 0.97, row_h - 0.004,
                          boxstyle="round,pad=0.002",
                          facecolor=bg_color, edgecolor="none",
                          transform=ax_leg.transAxes, zorder=0)
    ax_leg.add_patch(rect)

    medal_label = {1:"🥇", 2:"🥈", 3:"🥉"}.get(int(r.Rank), "  ")
    color       = rank_color(int(r.Rank))

    ax_leg.scatter([col_x[0] + 0.04], [y + 0.022], s=180, c=color,
                   edgecolors='black', linewidth=0.8,
                   transform=ax_leg.transAxes, zorder=5)
    ax_leg.text(col_x[0] + 0.04, y + 0.022, str(int(r.Rank)),
                ha='center', va='center', fontsize=6.5, fontweight='bold',
                transform=ax_leg.transAxes, zorder=6)
    ax_leg.text(col_x[1], y + 0.012, r.Area,
                fontsize=8.5, va='center', transform=ax_leg.transAxes)
    ax_leg.text(col_x[2], y + 0.012, f"{int(r.Breakeven_Students)} stu",
                fontsize=8.5, va='center', transform=ax_leg.transAxes)
    ax_leg.text(col_x[3], y + 0.012, f"PKR {int(r.Total_Monthly_PKR)//1000}k",
                fontsize=8.5, va='center', transform=ax_leg.transAxes)
    ax_leg.text(col_x[4], y + 0.012, f"{r.Combined_Score:.3f}",
                fontsize=8.5, va='center', fontweight='bold',
                transform=ax_leg.transAxes,
                color="#27ae60" if r.Combined_Score > 0.6 else "#e74c3c")

# Color legend
y_bottom = 0.86 - row_h * (len(ranked) + 1.5)
ax_leg.text(0.05, y_bottom, "Marker Colors: 🥇Gold=Rank1 | 🥈Silver=Rank2 | 🥉Bronze=Rank3 | "
            "🟢Rank4-6 | 🟠Rank7-9 | 🔴Rank10+",
            fontsize=7.5, va='top', transform=ax_leg.transAxes, color='grey',
            wrap=True)

plt.savefig(os.path.join(OUTPUT_DIR, "12_final_investment_map.png"), dpi=150, bbox_inches='tight')
plt.show()
print("✅ Final investment map saved")
print("Saved → 12_final_investment_map.png + top_locations_ranked.csv")

## Section 15 — Project Summary & Processed Dataset Export

In [ ]:
# ─────────────────────────────────────────────────────────────────
# EXPORT ALL PROCESSED DATASETS for submission
# ─────────────────────────────────────────────────────────────────
lahore_utm.to_file(os.path.join(OUTPUT_DIR, "lahore_boundary_utm.geojson"), driver='GeoJSON')
osm_lahore.to_file(os.path.join(OUTPUT_DIR, "osm_schools_processed.geojson"), driver='GeoJSON')

print("=" * 65)
print("  PROJECT SUMMARY — GIS School Site Selection, Lahore")
print("=" * 65)
print(f"  Schools analysed  : {len(all_schools):,} total")
print(f"    OSM (private)   : {len(osm_lahore):,}")
print(f"    PITB (govt)     : {len(pitb_lahore):,}")
print(f"  Population data   : WorldPop 2024, 100m resolution")
print(f"  Road network      : {len(G.nodes):,} nodes | {len(G.edges):,} edges")
print(f"  Property areas    : {len(zameen_df)} (Zameen.com + Graana, May 2026)")
print(f"  Student fee used  : PKR {FEE_PER_STUDENT:,}/month")
print()
print("  TOP 3 RECOMMENDED LOCATIONS:")
for _, r in ranked.head(3).iterrows():
    medal = {1:'🥇', 2:'🥈', 3:'🥉'}[int(r.Rank)]
    print(f"  {medal} #{int(r.Rank)} {r.Area}")
    print(f"      Monthly rent    : PKR {int(r.Avg_Rent_PKR):,}")
    print(f"      Total monthly   : PKR {int(r.Total_Monthly_PKR):,}")
    print(f"      Break-even      : {int(r.Breakeven_Students)} students")
    print(f"      Profit @150 stu : PKR {int(r.Profit_150st_PKR):,}/year")
    print(f"      Combined score  : {r.Combined_Score:.3f}")
    print()
print("=" * 65)
print(f"  All outputs saved to: {OUTPUT_DIR}")
print("=" * 65)

print("\n✅ All processed datasets exported:")
for f in sorted(os.listdir(OUTPUT_DIR)):
    size = os.path.getsize(os.path.join(OUTPUT_DIR, f))
    print(f"  {f:<55} ({size/1024:.1f} KB)")

## Section 16 — Future Vision: Industry-Grade Platform

This project is the analytical foundation for a **live web application** targeting entrepreneurs across Punjab who want data-driven location decisions for any micro-business.

### Planned Product Architecture
```
User Input
  └── Budget (PKR)
  └── Target Area in Lahore / Punjab
  └── Business Type (school / pharmacy / franchise / clinic)
  └── Max investment amount
        ↓
Real-Time Data Pipeline
  └── Zameen.com / OLX scraper → available buildings with rent
  └── WorldPop API → population data
  └── OSM / PITB → competitor locations
        ↓
Spatial Analysis Engine  ← This codebase
  └── KDE demand surface
  └── KD-Tree competition gap
  └── MCDA scoring
  └── Break-even calculation
        ↓
Output to User
  └── Pinpoint map of top 5 available buildings
  └── Per-building financial report
  └── Break-even enrollment / footfall / sales estimate

```

### Business Types Roadmap
| Type | Demand Layer | Competition Layer |
|---|---|---|
| 🏫 Private School | WorldPop children | OSM + PITB schools |
| 💊 Pharmacy | WorldPop population | OSM pharmacies |
| 🏪 Franchise outlet | Footfall / income | OSM shops |
| 🏥 Clinic | Population + income | OSM hospitals |

### Technical Stack
- **Backend:** Python FastAPI + PostGIS (spatial queries at scale)
- **Frontend:** React.js + Leaflet / Mapbox GL JS
- **Scraper:** Scrapy / Selenium for Zameen.com and OLX listings
- **ML layer:** XGBoost for enrolment / revenue prediction
- **Deployment:** AWS EC2 + RDS (PostgreSQL + PostGIS)
